**Pick a real dataset (your DATA, not a "training set")**
   → Filter/scope it down to something manageable
   → Write an EVAL SET (questions + expected answers, written by you)
   → Build naive baseline pipeline (chunk → embed → index → retrieve → generate)
   → Run eval set against baseline → get a score
   → Diagnose failures → apply ONE optimization technique
   → Re-run eval set → compare score
   → Repeat (diagnose → fix → re-score) for each technique
   → Consolidate validated techniques into final pipeline
   → Run eval set one last time → final score**

In [ ]:
!pip install openai
!pip install datasets pandas sentence-transformers qdrant-client rank-bm25 ragas

SyntaxError: invalid syntax (1717964538.py, line 1)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd
import json
import time

load_dotenv()  # reads OPENAI_API_KEY from .env

client = OpenAI()  # automatically picks up OPENAI_API_KEY from environment

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [ ]:
# Loading 2020 — the most recent year in EDGAR-CORPUS. Note: this is
# pandemic-year data, so expect risk factor sections especially to be
# longer and more repetitive than usual (lots of COVID-related language
# added across nearly every company). That's expected, not a data issue.
dataset = load_dataset("eloukas/edgar-corpus", "year_2020", trust_remote_code=True)
df = pd.DataFrame(dataset["train"])

print(f"Total filings in 2020: {len(df)}")
print(f"Columns available: {list(df.columns)}")

sample = df.iloc[0]
print(f"\nSample filing CIK: {sample['cik']}")
print(f"Sample filename: {sample['filename']}")
print(f"\nItem 1 (Business) preview:\n{sample['section_1'][:500]}")
print(f"\nItem 1A (Risk Factors) preview:\n{sample['section_1A'][:500]}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'eloukas/edgar-corpus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since eloukas/edgar-corpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'year_2020' at C:\Users\preet\.cache\huggingface\datasets\eloukas___edgar-corpus\year_2020\1.0.0\c2f9ada1db31915d6af4cc19f0ad9486cd0bab93c5c26bb32850e5a1f74f2bd7 (last modified on Fri Jun 19 22:50:04 2026).


Total filings in 2020: 5480
Columns available: ['filename', 'cik', 'year', 'section_1', 'section_1A', 'section_1B', 'section_2', 'section_3', 'section_4', 'section_5', 'section_6', 'section_7', 'section_7A', 'section_8', 'section_9', 'section_9A', 'section_9B', 'section_10', 'section_11', 'section_12', 'section_13', 'section_14', 'section_15']

Sample filing CIK: 718413
Sample filename: 718413_2020.htm

Item 1 (Business) preview:
Item 1. The Business
Organization and Operation
The Company. The Company was organized under the laws of the State of Vermont in 1982 and became a registered bank holding company under the Bank Holding Company Act of 1956, as amended, in October 1983 when it acquired all of the voting shares of the Bank, headquartered in Derby, Vermont. The Bank is the only subsidiary of the Company and principally all of the Company’s business operations are presently conducted through it. Therefore, the follow

Item 1A (Risk Factors) preview:
Item 1A. Risk Factors
Before dec

In [ ]:
# CIK numbers for companies you'll recognize, so you can sanity-check answers
# yourself later without needing to look anything up externally.
# These are real, public SEC EDGAR CIK identifiers.
KNOWN_COMPANIES = {
    "320193": "Apple",
    "1318605": "Tesla",
    "789019": "Microsoft",
    "1018724": "Amazon",
    "1652044": "Alphabet (Google)",
    "1326801": "Meta (Facebook)",
    "200406": "Johnson & Johnson",
    "19617": "JPMorgan Chase",
    "1045810": "Nvidia",
    "354950": "Home Depot",
}

# Reload from cache — this will be instant now since we already downloaded
# everything in the previous step.
dataset = load_dataset("eloukas/edgar-corpus", "year_2020", trust_remote_code=True)
df = pd.DataFrame(dataset["train"])

# cik in the dataframe is likely a string already, but we normalize just in
# case, since a type mismatch here would silently filter out everything.
df["cik"] = df["cik"].astype(str)

filtered = df[df["cik"].isin(KNOWN_COMPANIES.keys())].copy()
filtered["company_name"] = filtered["cik"].map(KNOWN_COMPANIES)

print(f"Matched {len(filtered)} of {len(KNOWN_COMPANIES)} known companies")
print(filtered[["cik", "company_name", "filename"]])

# Save this slice locally so every later script loads instantly from disk
# instead of re-filtering the full 5,480-row dataset each time.
filtered.to_parquet("filtered_2020_filings.parquet")
print("\nSaved to filtered_2020_filings.parquet")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'eloukas/edgar-corpus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since eloukas/edgar-corpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'year_2020' at C:\Users\preet\.cache\huggingface\datasets\eloukas___edgar-corpus\year_2020\1.0.0\c2f9ada1db31915d6af4cc19f0ad9486cd0bab93c5c26bb32850e5a1f74f2bd7 (last modified on Fri Jun 19 22:50:04 2026).


Matched 8 of 10 known companies
          cik       company_name          filename
438    789019          Microsoft   789019_2020.htm
504   1652044  Alphabet (Google)  1652044_2020.htm
1040  1018724             Amazon  1018724_2020.htm
2131  1326801    Meta (Facebook)  1326801_2020.htm
3359  1318605              Tesla  1318605_2020.htm
4085  1045810             Nvidia  1045810_2020.htm
4109   320193              Apple   320193_2020.htm
4683    19617     JPMorgan Chase    19617_2020.htm

Saved to filtered_2020_filings.parquet


In [ ]:

# Load the slice we already saved — instant, no re-downloading needed.
df = pd.read_parquet("filtered_2020_filings.parquet")

# Print a decent chunk (not just 500 chars) of Item 1A for each company so we
# have enough real text to write specific, checkable questions from.
for _, row in df.iterrows():
    print("=" * 80)
    print(f"{row['company_name']}  (CIK {row['cik']})")
    print("=" * 80)
    print(row["section_1A"][:1500])  # Risk Factors — usually the richest section
    print("\n")

Microsoft  (CIK 789019)
Item 1A
ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.
We face intense competition across all markets for our products and services, which may lead to lower revenue or operating margins.
Competition in the technology sector
Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and services. Our ability to remain co

In [ ]:

eval_set = [
    {
        "id": 1,
        "question": "What does Microsoft cite as a key competitive risk in its 2020 10-K?",
        "expected_answer": "Intense competition across all markets, especially from competitors with significant R&D resources or smaller specialized firms that can move faster.",
        "company": "Microsoft",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 2,
        "question": "What percentage of Alphabet's 2020 revenue came from advertising?",
        "expected_answer": "Over 80% of total revenues.",
        "company": "Alphabet (Google)",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 3,
        "question": "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?",
        "expected_answer": "Panasonic, which manufactures battery cells for Tesla at Gigafactory Nevada.",
        "company": "Tesla",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 4,
        "question": "What four markets does Nvidia say its GPU-based computing platforms address?",
        "expected_answer": "Gaming, Professional Visualization, Data Center, and Automotive.",
        "company": "Nvidia",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 5,
        "question": "Where should a reader look in Apple's 10-K alongside the risk factors section, per Apple's own cross-reference?",
        "expected_answer": "Item 7 (MD&A) and Item 8 (Financial Statements and Supplementary Data).",
        "company": "Apple",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 6,
        "question": "Both Tesla and JPMorgan Chase mention COVID-19 in their risk factors — how does the nature of the risk differ between the two?",
        "expected_answer": "Tesla's COVID risk centers on manufacturing/operational disruption (suspended factories, supplier impact like Panasonic); JPMorgan's centers on broader economic and regulatory harm to the financial services business.",
        "company": "Tesla, JPMorgan Chase",
        "section": "section_1A",
        "type": "comparative"
    },
    {
        "id": 7,
        "question": "How do Microsoft and Meta differ in how they introduce their risk factors section structurally?",
        "expected_answer": "Meta provides an explicit 'Summary Risk Factors' bulleted list upfront; Microsoft's introduction is narrative without an upfront bullet summary.",
        "company": "Microsoft, Meta",
        "section": "section_1A",
        "type": "comparative"
    },
    {
        "id": 8,
        "question": "What was Amazon's risk factors discussion in its 2020 10-K?",
        "expected_answer": "Not available — this section is empty in the dataset for Amazon's 2020 filing.",
        "company": "Amazon",
        "section": "section_1A",
        "type": "negative"
    },
    {
        "id": 9,
        "question": "What was Apple's stock price on a specific date in 2021?",
        "expected_answer": "Not enough information — out of corpus scope (no price data, no 2021 filings).",
        "company": "Apple",
        "section": "N/A",
        "type": "negative"
    },
]

with open("eval_set.json", "w") as f:
    json.dump(eval_set, f, indent=2)

print(f"Saved {len(eval_set)} eval questions to eval_set.json")

Saved 9 eval questions to eval_set.json


**Chunking**

In [ ]:
def fixed_size_chunk(text, chunk_size=800, overlap=100):
    if not text or pd.isna(text):
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []

for _, row in filtered.iterrows():
    text = row["section_1A"]
    if not text or pd.isna(text) or len(text.strip()) == 0:
        continue
    chunks = fixed_size_chunk(text)
    for chunk_text in chunks:
        all_chunks.append({
            "text": chunk_text,
            "company": row["company_name"],
            "cik": row["cik"],
            "section": "section_1A"
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Example chunk:\n{all_chunks[0]['text'][:300]}")

Total chunks created: 893
Example chunk:
Item 1A
ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.
We face int


In [ ]:
#step 2 convert to embeddings and store in vector db
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Loads the embedding model. First run downloads it (~400MB), then it's
# cached locally. bge-base-en-v1.5 is a free, well-regarded open model —
# no API key needed, runs entirely on your CPU/GPU.
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

# Embed every chunk's text. This converts each chunk into a 768-dimensional
# vector — a list of numbers positioned so that semantically similar text
# ends up close together in that 768-dimensional space.
texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True)

print(f"Embedded {len(embeddings)} chunks")
print(f"Each embedding has {len(embeddings[0])} dimensions")

# Set up Qdrant in-memory (no Docker needed, as decided earlier).
client = QdrantClient(":memory:")

# A "collection" in Qdrant is like a table — we define its name and the
# shape/distance-metric of vectors it will hold.
client.create_collection(
    collection_name="risk_factors_v0",
    vectors_config=VectorParams(size=len(embeddings[0]), distance=Distance.COSINE)
)

# Upload every chunk's vector + its original text/metadata as a "point".
# Cosine distance measures the angle between two vectors — the standard
# choice for text embeddings, since it cares about direction (meaning)
# rather than raw magnitude.
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=all_chunks[i]  # keeps text, company, cik, section alongside the vector
    )
    for i in range(len(all_chunks))
]
client.upsert(collection_name="risk_factors_v0", points=points)

print(f"Indexed {len(points)} chunks into Qdrant")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedded 893 chunks
Each embedding has 768 dimensions
Indexed 893 chunks into Qdrant


In [ ]:
# step 3 Retrieval and generation

load_dotenv()
llm = OpenAI()

def retrieve(query, top_k=3):
    """
    Embeds the query with the SAME model used for the chunks (critical —
    query and chunks must live in the same vector space to be comparable),
    then asks Qdrant for the top_k closest chunks by cosine similarity.
    """
    query_vector = embedder.encode(query).tolist()
    results = client.query_points(
        collection_name="risk_factors_v0",
        query=query_vector,
        limit=top_k
    )
    return results.points  # newer qdrant-client wraps results in .points

def generate_answer(query, retrieved_chunks):
    """
    Builds a grounded prompt: the retrieved context first, then the
    question, with explicit instructions to only use the given context
    and to say so honestly if the answer isn't there. This last
    instruction is the cheapest hallucination defense we have.
    """
    context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in retrieved_chunks
    ])

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0  # deterministic-ish output, important for fair eval comparisons later
    )
    return response.choices[0].message.content

# Quick manual test before running the full eval set
test_query = "What percentage of Alphabet's 2020 revenue came from advertising?"
retrieved = retrieve(test_query)
answer = generate_answer(test_query, retrieved)

print(f"Question: {test_query}")
print(f"\nRetrieved from: {[r.payload['company'] for r in retrieved]}")
print(f"\nAnswer: {answer}")

   


Question: What percentage of Alphabet's 2020 revenue came from advertising?

Retrieved from: ['Nvidia', 'Alphabet (Google)', 'Alphabet (Google)']

Answer: I don't have enough information to answer that.


In [ ]:
for i, r in enumerate(retrieved):
    print(f"--- Result {i+1} (score: {r.score:.4f}) ---")
    print(f"Company: {r.payload['company']}")
    print(f"Text: {r.payload['text'][:400]}")
    print()

--- Result 1 (score: 0.6542) ---
Company: Nvidia
Text: actured, assembled, tested and packaged by third parties located outside of the United States. We also generate a significant portion of our revenue from sales outside the United States. We allocate revenue to individual countries based on the location to which the products are initially billed even if our customers’ revenue is attributable to end customers that are located in a different location

--- Result 2 (score: 0.6326) ---
Company: Alphabet (Google)
Text: Our operating results may also suffer if our products and services are not responsive to the needs of our users, advertisers, publishers, customers, and content providers. As technologies continue to develop, our competitors may be able to offer experiences that are, or that are seen to be, substantially similar to or better than ours. This
Alphabet Inc.
may force us to compete in different ways a

--- Result 3 (score: 0.6258) ---
Company: Alphabet (Google)
Text: clude, bu

**This shows a basic Rag failure with fixed chunk as it was not able to identify properly the relevant ones and lets try running on the entire eval set**

In [ ]:

with open("eval_set.json") as f:
    eval_set = json.load(f)

baseline_results = []

for item in eval_set:
    query = item["question"]
    retrieved = retrieve(query, top_k=3)
    answer = generate_answer(query, retrieved)

    baseline_results.append({
        "id": item["id"],
        "question": query,
        "expected_answer": item["expected_answer"],
        "generated_answer": answer,
        "retrieved_companies": [r.payload["company"] for r in retrieved],
        "type": item["type"]
    })

    print(f"Q{item['id']} [{item['type']}]: {query}")
    print(f"  Expected:  {item['expected_answer'][:100]}")
    print(f"  Generated: {answer[:150]}")
    print(f"  Retrieved from: {[r.payload['company'] for r in retrieved]}")
    print()

with open("baseline_v0_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print(f"Saved {len(baseline_results)} results to baseline_v0_results.json")

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Expected:  Intense competition across all markets, especially from competitors with significant R&D resources o
  Generated: Microsoft cites its increasing focus on cloud-based services as a key competitive risk in its 2020 10-K.
  Retrieved from: ['Alphabet (Google)', 'Microsoft', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Expected:  Over 80% of total revenues.
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Nvidia', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Expected:  Panasonic, which manufactures battery cells for Tesla at Gigafactory Nevada.
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets d

In [ ]:
scored_baseline = {
    "version": "v0_naive_fixed_chunking_dense_only",
    "score": "4/9",
    "accuracy": 4/9,
    "failures": [1, 2, 3, 6, 7],
    "successes": [4, 5, 8, 9],
    "notes": "Single-company factual misses caused by imprecise chunk-level retrieval. Both comparative questions failed entirely — single dense query can't reliably surface multiple companies at once."
}

with open("baseline_v0_score.json", "w") as f:
    json.dump(scored_baseline, f, indent=2)

print("Baseline locked: 4/9 (44%)")

Baseline locked: 4/9 (44%)


In [ ]:
!pip install langchain langchain-text-splitters

In [ ]:
#Now lets try using recursivetextsplitter 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client.models import VectorParams, Distance, PointStruct

def build_chunks_recursive(filtered_df, chunk_size=800, chunk_overlap=100):
    """
    Unlike our manual fixed_size_chunk, this tries to split on paragraph
    breaks first, then sentences, then words — only cutting mid-word as an
    absolute last resort. Should fix the 'We face int...' mid-sentence cut
    we saw in v0.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = []
    for _, row in filtered_df.iterrows():
        text = row["section_1A"]
        if not text or pd.isna(text) or len(text.strip()) == 0:
            continue
        for chunk_text in splitter.split_text(text):
            chunks.append({
                "text": chunk_text,
                "company": row["company_name"],
                "cik": row["cik"],
                "section": "section_1A"
            })
    return chunks


def index_chunks(chunks, collection_name):
    """Embeds a list of chunks and loads them into a fresh Qdrant collection."""
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts, show_progress_bar=True)

    # Recreate the collection fresh each time so old runs don't bleed in
    client.delete_collection(collection_name=collection_name) if client.collection_exists(collection_name) else None
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=len(embeddings[0]), distance=Distance.COSINE)
    )
    points = [
        PointStruct(id=i, vector=embeddings[i].tolist(), payload=chunks[i])
        for i in range(len(chunks))
    ]
    client.upsert(collection_name=collection_name, points=points)
    return len(points)


def retrieve_from(collection_name, query, top_k=3):
    query_vector = embedder.encode(query).tolist()
    results = client.query_points(collection_name=collection_name, query=query_vector, limit=top_k)
    return results.points


def run_eval(collection_name, eval_set, top_k=3):
    """Runs the full eval set against a given collection and returns results."""
    results = []
    for item in eval_set:
        retrieved = retrieve_from(collection_name, item["question"], top_k)
        answer = generate_answer(item["question"], retrieved)
        results.append({
            "id": item["id"],
            "type": item["type"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "generated_answer": answer,
            "retrieved_companies": [r.payload["company"] for r in retrieved]
        })
    return results

In [ ]:
recursive_chunks = build_chunks_recursive(filtered)
print(f"Recursive chunking produced {len(recursive_chunks)} chunks (v0 had {len(all_chunks)})")
print(f"Example chunk:\n{recursive_chunks[0]['text'][:300]}")

n_indexed = index_chunks(recursive_chunks, "risk_factors_recursive")
print(f"Indexed {n_indexed} chunks")

recursive_results = run_eval("risk_factors_recursive", eval_set)
for r in recursive_results:
    print(f"Q{r['id']} [{r['type']}]: {r['question']}")
    print(f"  Generated: {r['generated_answer'][:150]}")
    print(f"  Retrieved from: {r['retrieved_companies']}")
    print()

Recursive chunking produced 1129 chunks (v0 had 893)
Example chunk:
Item 1A
ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.
We face int


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Indexed 1129 chunks
Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Apple', 'Alphabet (Google)', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Alphabet (Google)', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Nvidia says its GPU-based computing platforms address the following four markets: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple']

Q5 [fact

In [ ]:
retrieved_q2 = retrieve_from("risk_factors_recursive", "What percentage of Alphabet's 2020 revenue came from advertising?", top_k=3)
for r in retrieved_q2:
    print(f"Score: {r.score:.4f}")
    print(r.payload["text"][:400])
    print()

Score: 0.6842
Alphabet Inc.
Our international operations expose us to additional risks that could harm our business, our financial condition, and operating results.
Our international operations are significant to our revenues and net income, and we plan to continue to grow internationally. International revenues accounted for approximately 53% of our consolidated revenues in 2020. In addition to risks described

Score: 0.6671
We generated over 80% of total revenues from the display of ads online in 2020. Many of our advertisers, companies that distribute our products and services, digital publishers, and content providers can terminate their contracts with us at any time. These partners may not continue to do business with us if we do not create more value (such as increased numbers of users or customers, new sales lea

Score: 0.6529
•Our ability to attract user and/or customer adoption of, and generate significant revenues from, new products, services, and technologies in which we hav

In [ ]:
retrieved_q2 = retrieve_from("risk_factors_recursive", "What percentage of Alphabet's 2020 revenue came from advertising?", top_k=3)

context = "\n\n---\n\n".join([
    f"[{r.payload['company']}]: {r.payload['text']}"
    for r in retrieved_q2
])

prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: What percentage of Alphabet's 2020 revenue came from advertising?

Answer:"""

print("=== FULL PROMPT SENT TO LLM ===")
print(prompt)
print("\n=== LLM RESPONSE ===")
response = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)
print(response.choices[0].message.content)

=== FULL PROMPT SENT TO LLM ===
Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
[Alphabet (Google)]: Alphabet Inc.
Our international operations expose us to additional risks that could harm our business, our financial condition, and operating results.
Our international operations are significant to our revenues and net income, and we plan to continue to grow internationally. International revenues accounted for approximately 53% of our consolidated revenues in 2020. In addition to risks described elsewhere in this section, our international operations expose us to other risks, including the following:
•Restrictions on foreign ownership and investments, and stringent foreign exchange controls that might prevent us from repatriating cash earned in countries outside the U.S.

---

[Alphabet (Google)]: We generated over 80% of total revenues fr

In [ ]:
# Reorder retrieved chunks by score, ascending, so the BEST match lands
# immediately before the question — closest to where attention is strongest.
reordered = sorted(retrieved_q2, key=lambda r: r.score)  # lowest score first, highest last

context_reordered = "\n\n---\n\n".join([
    f"[{r.payload['company']}]: {r.payload['text']}"
    for r in reordered
])

prompt_reordered = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context_reordered}

Question: What percentage of Alphabet's 2020 revenue came from advertising?

Answer:"""

response = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_reordered}],
    temperature=0
)
print(response.choices[0].message.content)

Over 80% of total revenues came from the display of ads online in 2020.


In [ ]:
def generate_answer(query, retrieved_chunks):
    """
    Builds a grounded prompt. Chunks are sorted lowest-score-to-highest
    before insertion, so the single most relevant chunk sits immediately
    before the question — closest to where the model's attention is
    strongest, addressing the 'lost in the middle' problem we proved out
    on the Alphabet 80% question.
    """
    reordered = sorted(retrieved_chunks, key=lambda r: r.score)

    context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in reordered
    ])

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
recursive_results_v2 = run_eval("risk_factors_recursive", eval_set)
for r in recursive_results_v2:
    print(f"Q{r['id']} [{r['type']}]: {r['question']}")
    print(f"  Generated: {r['generated_answer'][:150]}")
    print(f"  Retrieved from: {r['retrieved_companies']}")
    print()

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Apple', 'Alphabet (Google)', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: Over 80% of total revenues came from the display of ads online in 2020.
  Retrieved from: ['Alphabet (Google)', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple']

Q5 [factual]: Where should a reader look in Apple's 10-K alongside the risk factors se

In [ ]:
#testing

def test_ordering(chunks, label):
    context = "\n\n---\n\n".join([f"[{r.payload['company']}]: {r.payload['text']}" for r in chunks])
    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: What percentage of Alphabet's 2020 revenue came from advertising?

Answer:"""
    response = llm.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    print(f"{label}: {response.choices[0].message.content}")

# Original order, as Qdrant returned it (descending by score)
test_ordering(retrieved_q2, "Original order (Qdrant default, descending score)")

# Correct version: TRUE highest score last
true_sorted = sorted(retrieved_q2, key=lambda r: r.score)  # this is actually correct ascending — re-verify
test_ordering(true_sorted, "Ascending score sort")

# Run original order again to check for non-determinism
test_ordering(retrieved_q2, "Original order, repeated")

Original order (Qdrant default, descending score): I don't have enough information to answer that.
Ascending score sort: Over 80% of total revenues came from the display of ads online in 2020.
Original order, repeated: I don't have enough information to answer that.


In [ ]:
checkpoint = {
    "version": "recursive_chunking_plus_reorder",
    "score": "5/9",
    "accuracy": 5/9,
    "failures": [1, 3, 6, 7],
    "successes": [2, 4, 5, 8, 9],
    "notes": "Reorder fixed Q2 (proven reproducible via repeated test). Q1/Q3 still fail despite correct company being retrieved — likely the specific needed sentence isn't in top-3 chunks. Q6/Q7 (comparative) still fail entirely — single dense query structurally can't pull both companies into one result set."
}

with open("checkpoint_v2_score.json", "w") as f:
    json.dump(checkpoint, f, indent=2)

print("Checkpoint saved: 5/9")

Checkpoint saved: 5/9


The core problem: a single global top-3 search lets one company's chunks dominate, crowding out the other entirely. The fix is metadata filtering combined with query decomposition — detect which companies a question mentions, retrieve separately per company, then merge before generation.

In [ ]:
#Now lets do another improvisation 
COMPANY_NAMES = filtered["company_name"].unique().tolist()
print(COMPANY_NAMES)

def detect_companies(query, company_names):
    """
    Simple keyword match: check which known company names appear in the
    question text. Not fancy NLP — just substring matching, which is
    enough for our 8-company corpus. A production system would handle
    aliases (e.g. 'Google' vs 'Alphabet') more robustly.
    """
    found = [name for name in company_names if name.split()[0].lower() in query.lower()]
    return found

# Quick test
print(detect_companies("Both Tesla and JPMorgan Chase mention COVID-19...", COMPANY_NAMES))


['Microsoft', 'Alphabet (Google)', 'Amazon', 'Meta (Facebook)', 'Tesla', 'Nvidia', 'Apple', 'JPMorgan Chase']
['Tesla', 'JPMorgan Chase']


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def retrieve_with_decomposition(collection_name, query, top_k=3):
    """
    If the question mentions multiple known companies, retrieve top_k
    chunks SEPARATELY for each company (using a metadata filter so each
    company gets a fair, guaranteed share of the results), then merge.
    Otherwise, fall back to normal single-pass retrieval.
    """
    companies = detect_companies(query, COMPANY_NAMES)
    query_vector = embedder.encode(query).tolist()

    if len(companies) >= 2:
        all_results = []
        for company in companies:
            results = client.query_points(
                collection_name=collection_name,
                query=query_vector,
                query_filter=Filter(
                    must=[FieldCondition(key="company", match=MatchValue(value=company))]
                ),
                limit=top_k
            )
            all_results.extend(results.points)
        return all_results
    else:
        results = client.query_points(collection_name=collection_name, query=query_vector, limit=top_k)
        return results.points

In [ ]:
#test 1
q6 = "Both Tesla and JPMorgan Chase mention COVID-19 in their risk factors — how does the nature of the risk differ between the two?"
retrieved_q6 = retrieve_with_decomposition("risk_factors_recursive", q6)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_q6]}")
answer_q6 = generate_answer(q6, retrieved_q6)
print(f"Answer: {answer_q6}")

Retrieved from: ['Tesla', 'Tesla', 'Tesla', 'JPMorgan Chase', 'JPMorgan Chase', 'JPMorgan Chase']
Answer: The nature of the risk related to COVID-19 differs between Tesla and JPMorgan Chase in that Tesla focuses on the impact of the pandemic on macroeconomic conditions, manufacturing operations, supply chains, and consumer behavior, which could affect its growth and market share in the automotive industry. In contrast, JPMorgan Chase emphasizes the potential for increased governmental scrutiny, negative publicity, and litigation risks associated with its participation in government programs aimed at supporting those affected by the pandemic, as well as the broader negative effects on its business and financial condition.


In [ ]:
#test 2 
q7 = "How do Microsoft and Meta differ in how they introduce their risk factors section structurally?"
retrieved_q7 = retrieve_with_decomposition("risk_factors_recursive", q7)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_q7]}")
answer_q7 = generate_answer(q7, retrieved_q7)
print(f"Answer: {answer_q7}")

Retrieved from: ['Microsoft', 'Microsoft', 'Microsoft', 'Meta (Facebook)', 'Meta (Facebook)', 'Meta (Facebook)']
Answer: Microsoft presents its risk factors section with a more detailed introduction that emphasizes the centrality of data insights to their services and the potential regulatory constraints affecting their operations. In contrast, Meta introduces its risk factors section more generally, stating that their business is subject to various risks without delving into specific themes or areas of focus. Additionally, Meta lists specific risks related to product offerings, while Microsoft discusses broader implications of data management and operational challenges.


In [ ]:
def run_eval_v2(collection_name, eval_set, top_k=3):
    results = []
    for item in eval_set:
        retrieved = retrieve_with_decomposition(collection_name, item["question"], top_k)
        answer = generate_answer(item["question"], retrieved)
        results.append({
            "id": item["id"],
            "type": item["type"],
            "question": item["question"],
            "generated_answer": answer,
            "retrieved_companies": [r.payload["company"] for r in retrieved]
        })
        print(f"Q{item['id']} [{item['type']}]: {item['question']}")
        print(f"  Generated: {answer[:150]}")
        print(f"  Retrieved from: {[r.payload['company'] for r in retrieved]}")
        print()
    return results

decomposition_results = run_eval_v2("risk_factors_recursive", eval_set)

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Apple', 'Alphabet (Google)', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: Over 80% of total revenues came from the display of ads online in 2020.
  Retrieved from: ['Alphabet (Google)', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple']

Q5 [factual]: Where should a reader look in Apple's 10-K alongside the risk factors se

In [ ]:
checkpoint_v3 = {
    "version": "recursive_chunking_reorder_metadata_filter",
    "score": "7/9",
    "accuracy": 7/9,
    "failures": [1, 3],
    "successes": [2, 4, 5, 6, 7, 8, 9],
    "notes": "Metadata filtering (per-company retrieval) fixed both comparative questions completely. Q1/Q3 remain the only failures — both are single-company factual questions where the right company IS retrieved but the specific answer sentence isn't in the top-3 chunks."
}
with open("checkpoint_v3_score.json", "w") as f:
    json.dump(checkpoint_v3, f, indent=2)
print("Checkpoint saved: 7/9")

Checkpoint saved: 7/9


Q1 and Q3 are both the same failure type we diagnosed way back with the original Alphabet case — the right company gets retrieved, but the precise sentence holding the answer doesn't make the top 3. Look at Q3 specifically: it's asking about "Panasonic" by name — a literal proper noun. Dense embeddings are notoriously weak at exact-term matching like this, because "Panasonic" doesn't have a strong semantic relationship to the rest of the question's wording ("supplier," "COVID-19 disruption") the way a topically-similar paragraph might score higher just by being more generally about COVID and operations.

In [ ]:
#Lets try with hybrid search using BM25 keyword alongside dense  vector search
#Build BM25 index over the same chunks 
from rank_bm25 import BM25Okapi

# BM25 needs tokenized text — simplest reasonable approach: lowercase and
# split on whitespace. Production systems often use a proper tokenizer,
# but this is sufficient for our scale and purpose.
def tokenize(text):
    return text.lower().split()

bm25_corpus = [tokenize(c["text"]) for c in recursive_chunks]
bm25_index = BM25Okapi(bm25_corpus)

print(f"BM25 index built over {len(bm25_corpus)} chunks")

# Quick sanity check: does BM25 find the Panasonic chunk for a Panasonic query?
test_scores = bm25_index.get_scores(tokenize("Panasonic supplier COVID disruption"))
top_idx = test_scores.argsort()[::-1][:3]
for idx in top_idx:
    print(f"Score: {test_scores[idx]:.2f} | Company: {recursive_chunks[idx]['company']}")
    print(f"  {recursive_chunks[idx]['text'][:200]}")

BM25 index built over 1129 chunks
Score: 9.78 | Company: Tesla
  Our plan to grow the volume and profitability of our vehicles and energy storage products depends on significant lithium-ion battery cell production by our partner Panasonic at Gigafactory Nevada. Alt
Score: 8.15 | Company: Tesla
  We are dependent on the continued supply of lithium-ion battery cells for our vehicles and energy storage products, and we will require substantially more cells to grow our business according to our p
Score: 7.25 | Company: Tesla
  such as battery modules and packs incorporating the cells produced by Panasonic for Model 3 and Model Y and drive units (including to support Gigafactory Shanghai production), at Gigafactory Nevada, a


In [ ]:
def hybrid_retrieve(collection_name, query, top_k=3, k_rrf=60):
    """
    Runs dense (Qdrant) and lexical (BM25) search separately, then fuses
    their RANKS using Reciprocal Rank Fusion: each chunk's fused score is
    the sum of 1/(k_rrf + rank) across both result lists. A chunk that
    ranks well in EITHER method gets a meaningful boost.
    """
    query_vector = embedder.encode(query).tolist()
    dense_results = client.query_points(collection_name=collection_name, query=query_vector, limit=20).points

    bm25_scores = bm25_index.get_scores(tokenize(query))
    bm25_ranked_ids = bm25_scores.argsort()[::-1][:20]

    fused_scores = {}
    payload_lookup = {}

    for rank, r in enumerate(dense_results):
        fused_scores[r.id] = fused_scores.get(r.id, 0) + 1 / (k_rrf + rank)
        payload_lookup[r.id] = r.payload

    for rank, idx in enumerate(bm25_ranked_ids):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0) + 1 / (k_rrf + rank)
        if idx not in payload_lookup:
            payload_lookup[idx] = recursive_chunks[idx]

    top_ids = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    class FusedResult:
        def __init__(self, payload, score):
            self.payload = payload
            self.score = score

    return [FusedResult(payload_lookup[cid], score) for cid, score in top_ids]


# Test directly on Q3
real_q3 = "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?"
test_scores = bm25_index.get_scores(tokenize(real_q3))
top_idx = test_scores.argsort()[::-1][:5]
for idx in top_idx:
    print(f"Score: {test_scores[idx]:.2f} | Company: {recursive_chunks[idx]['company']}")
    print(f"  {recursive_chunks[idx]['text'][:200]}")

Score: 15.07 | Company: Tesla
  and other costs in the State of New York during the 10-year period beginning April 30, 2018. As we temporarily suspended most of our manufacturing operations at Gigafactory New York pursuant to a New 
Score: 12.77 | Company: JPMorgan Chase
  The U.K.’s departure from the EU, which is commonly referred to as “Brexit,” was completed on December 31, 2020. The Trade and Cooperation Agreement entered into between the U.K. and the EU in Decembe
Score: 12.74 | Company: Meta (Facebook)
  competition, and consumer protection authorities in the European Union and other jurisdictions have initiated actions, investigations, or administrative orders seeking to restrict the ways in which we
Score: 12.62 | Company: Meta (Facebook)
  and modified consent order to resolve the FTC inquiry, which was approved by the federal court and took effect in April 2020. Among other matters, our settlement with the FTC required us to pay a pena
Score: 12.57 | Company: Meta (Facebook)

In [ ]:
#Lets try with HyDE
def hyde_retrieve(collection_name, query, top_k=3):
    # Ask the LLM to draft a plausible hypothetical answer to the question,
    # using general domain knowledge of 10-K filing language — not
    # grounded in our corpus at all, just a guess at what the real answer
    # would probably sound like.
    hyde_prompt = f"Write a one-sentence hypothetical excerpt from a company's 10-K risk factors section that would answer this question: {query}"
    hyde_response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": hyde_prompt}],
        temperature=0.3
    )
    hypothetical_answer = hyde_response.choices[0].message.content
    print(f"Hypothetical answer used for embedding: {hypothetical_answer}\n")

    # Embed the HYPOTHETICAL ANSWER, not the original question
    query_vector = embedder.encode(hypothetical_answer).tolist()
    results = client.query_points(collection_name=collection_name, query=query_vector, limit=top_k)
    return results.points

retrieved_hyde = hyde_retrieve("risk_factors_recursive", real_q3)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_hyde]}")
for r in retrieved_hyde:
    print(f"  {r.payload['text'][:150]}")

Hypothetical answer used for embedding: In our risk factors section, we acknowledge that disruptions caused by the COVID-19 pandemic have impacted our supply chain, particularly citing our reliance on Panasonic for battery cell production, which faced operational challenges during the global health crisis.

Retrieved from: ['Apple', 'Microsoft', 'Apple']
  The COVID-19 pandemic and the measures taken by many countries in response have adversely affected and could in the future materially adversely impact
  In the third and fourth quarters of fiscal year 2020, we have experienced adverse impacts to our supply chain, a slowdown in transactional licensing, 
  The Company is continuing to monitor the situation and take appropriate actions in accordance with the recommendations and requirements of relevant au


Even HYde has failed to retrieve the right one as above since every company has the covid 19 similar use cases and it failed to retrieve the relevant ones 

In [ ]:
final_checkpoint = {
    "version": "all_techniques_combined",
    "score": "7/9",
    "accuracy": 7/9,
    "remaining_failures": [1, 3],
    "failure_diagnosis": "Q1 and Q3 both require recalling a specific named entity (a competitor type, a supplier name) from within text that has substantial topical overlap with OTHER companies' chunks, specifically because 2020 is pandemic-year data and COVID-disruption boilerplate is repeated near-identically across the corpus. Tried: better chunking, reordering, metadata filtering, hybrid search (BM25), and HyDE — none fully resolved it. Root cause appears to be embedding dilution of specific entities within generically-similar boilerplate, a known weakness of dense retrieval.",
    "techniques_validated": [
        "Recursive chunking (no measured score change alone, but cleaner chunks)",
        "Context reordering by relevance score (4/9 -> 5/9, fixed Q2)",
        "Metadata filtering for multi-entity questions (5/9 -> 7/9, fixed Q6 and Q7 completely)",
        "Hybrid search and HyDE (no improvement on this corpus/eval set, documented why)"
    ]
}
with open("final_checkpoint_day1.json", "w") as f:
    json.dump(final_checkpoint, f, indent=2)
print("Day 1 results locked: 7/9, with documented analysis of remaining failures")

Day 1 results locked: 7/9, with documented analysis of remaining failures


Reranking

In [ ]:
from sentence_transformers import CrossEncoder

# Same model, loaded through sentence-transformers' CrossEncoder class
# instead of FlagEmbedding — more stable, fewer dependency conflicts,
# and you already have the library installed.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank_retrieve(collection_name, query, candidate_k=15, final_k=3):
    query_vector = embedder.encode(query).tolist()
    candidates = client.query_points(collection_name=collection_name, query=query_vector, limit=candidate_k).points
    
    pairs = [[query, c.payload["text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)  # CrossEncoder uses .predict(), not .compute_score()

    scored = list(zip(candidates, rerank_scores))
    scored.sort(key=lambda x: x[1], reverse=True)

    class RerankedResult:
        def __init__(self, payload, score):
            self.payload = payload
            self.score = score

    return [RerankedResult(c.payload, score) for c, score in scored[:final_k]]


real_q3 = "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?"
retrieved_rerank = rerank_retrieve("risk_factors_recursive", real_q3, candidate_k=15, final_k=3)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_rerank]}")
for r in retrieved_rerank:
    print(f"  Score: {r.score:.4f} | {r.payload['text'][:150]}")

answer_rerank = generate_answer(real_q3, retrieved_rerank)
print(f"\nAnswer: {answer_rerank}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Retrieved from: ['Apple', 'Alphabet (Google)', 'Tesla']
  Score: 0.0323 | The COVID-19 pandemic and the measures taken by many countries in response have adversely affected and could in the future materially adversely impact
  Score: 0.0202 | The occurrence of a natural disaster or pandemic (including COVID-19), closure of a facility, or other unanticipated problems at, or impacting, our da
  Score: 0.0156 | Our products contain thousands of parts that we purchase globally from hundreds of mostly single-source direct suppliers, generally without long-term 

Answer: I don't have enough information to answer that.


**As we see in the above Panasonic related text is still not retrieved for Q3 .Lets debug**

In [ ]:
query_vector = embedder.encode(real_q3).tolist()
candidates = client.query_points(collection_name="risk_factors_recursive", query=query_vector, limit=15).points

print(f"Checking if any of the 15 candidates mention Panasonic:\n")
found = False
for i, c in enumerate(candidates):
    has_panasonic = "panasonic" in c.payload["text"].lower()
    if has_panasonic:
        found = True
    print(f"{i+1}. [{c.payload['company']}] Panasonic mentioned: {has_panasonic}")

print(f"\nPanasonic chunk present in candidate pool: {found}")

Checking if any of the 15 candidates mention Panasonic:

1. [Tesla] Panasonic mentioned: False
2. [Meta (Facebook)] Panasonic mentioned: False
3. [Alphabet (Google)] Panasonic mentioned: False
4. [Apple] Panasonic mentioned: False
5. [Apple] Panasonic mentioned: False
6. [Tesla] Panasonic mentioned: False
7. [Microsoft] Panasonic mentioned: False
8. [Apple] Panasonic mentioned: False
9. [Meta (Facebook)] Panasonic mentioned: False
10. [Meta (Facebook)] Panasonic mentioned: False
11. [Meta (Facebook)] Panasonic mentioned: False
12. [Apple] Panasonic mentioned: False
13. [Apple] Panasonic mentioned: False
14. [Alphabet (Google)] Panasonic mentioned: False
15. [Meta (Facebook)] Panasonic mentioned: False

Panasonic chunk present in candidate pool: False


**The above result shows up that the issue is not with RRF or reranking .Its a recall problem mainly with retrieval depth**

In [ ]:
# Search much wider — essentially the whole index — to find where the
# Panasonic chunk actually ranks for this exact question.
wide_results = client.query_points(
    collection_name="risk_factors_recursive",
    query=query_vector,
    limit=len(recursive_chunks)  # the entire indexed set
).points

panasonic_rank = None
for i, r in enumerate(wide_results):
    if "panasonic" in r.payload["text"].lower():
        panasonic_rank = i + 1  # 1-indexed for readability
        print(f"First Panasonic-mentioning chunk found at rank: {panasonic_rank} (score: {r.score:.4f})")
        print(f"  [{r.payload['company']}] {r.payload['text'][:200]}")
        break

if panasonic_rank is None:
    print("No chunk mentioning Panasonic was found anywhere in dense search results.")

First Panasonic-mentioning chunk found at rank: 29 (score: 0.6203)
  [Tesla] We temporarily suspended operations at each of our manufacturing facilities worldwide for a part of the first half of 2020. Some of our suppliers and partners also experienced temporary suspensions be


In [ ]:
retrieved_rerank_wide = rerank_retrieve("risk_factors_recursive", real_q3, candidate_k=35, final_k=3)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_rerank_wide]}")
for r in retrieved_rerank_wide:
    print(f"  Score: {r.score:.4f} | {r.payload['text'][:150]}")

answer_rerank_wide = generate_answer(real_q3, retrieved_rerank_wide)
print(f"\nAnswer: {answer_rerank_wide}")

Retrieved from: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)']
  Score: 0.0323 | The COVID-19 pandemic and the measures taken by many countries in response have adversely affected and could in the future materially adversely impact
  Score: 0.0260 | On March 11, 2020, the World Health Organization declared the outbreak of a strain of novel coronavirus disease, COVID-19, to be a global pandemic. Th
  Score: 0.0202 | The occurrence of a natural disaster or pandemic (including COVID-19), closure of a facility, or other unanticipated problems at, or impacting, our da

Answer: I don't have enough information to answer that.


In [ ]:
#diagnostic test as why it still not retrieving the Q3 factual answer
query_vector = embedder.encode(real_q3).tolist()
candidates_35 = client.query_points(collection_name="risk_factors_recursive", query=query_vector, limit=35).points

pairs = [[real_q3, c.payload["text"]] for c in candidates_35]
scores_35 = reranker.predict(pairs)

scored_35 = sorted(zip(candidates_35, scores_35), key=lambda x: x[1], reverse=True)

for i, (c, score) in enumerate(scored_35):
    has_panasonic = "panasonic" in c.payload["text"].lower()
    marker = "  <<< PANASONIC CHUNK" if has_panasonic else ""
    print(f"{i+1}. score={score:.4f} [{c.payload['company']}]{marker}")

1. score=0.0323 [Apple]
2. score=0.0260 [JPMorgan Chase]
3. score=0.0202 [Alphabet (Google)]
4. score=0.0156 [Tesla]
5. score=0.0082 [Tesla]  <<< PANASONIC CHUNK
6. score=0.0075 [Meta (Facebook)]
7. score=0.0071 [Meta (Facebook)]
8. score=0.0070 [Alphabet (Google)]
9. score=0.0059 [Tesla]
10. score=0.0051 [Nvidia]
11. score=0.0050 [JPMorgan Chase]
12. score=0.0046 [Meta (Facebook)]
13. score=0.0045 [Apple]
14. score=0.0044 [Alphabet (Google)]
15. score=0.0042 [Apple]
16. score=0.0037 [Meta (Facebook)]
17. score=0.0037 [Meta (Facebook)]
18. score=0.0027 [Meta (Facebook)]
19. score=0.0023 [Apple]
20. score=0.0022 [Tesla]
21. score=0.0022 [Tesla]
22. score=0.0022 [Apple]
23. score=0.0021 [Meta (Facebook)]
24. score=0.0020 [Microsoft]
25. score=0.0018 [Apple]
26. score=0.0017 [Meta (Facebook)]
27. score=0.0014 [Meta (Facebook)]
28. score=0.0013 [Meta (Facebook)]
29. score=0.0012 [Meta (Facebook)]
30. score=0.0010 [Apple]
31. score=0.0006 [Alphabet (Google)]
32. score=0.0004 [Nvidia]
33. sc

In [ ]:
retrieved_final = rerank_retrieve("risk_factors_recursive", real_q3, candidate_k=35, final_k=5)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_final]}")

answer_final = generate_answer(real_q3, retrieved_final)
print(f"\nAnswer: {answer_final}")

Retrieved from: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)', 'Tesla', 'Tesla']

Answer: Panasonic


In [ ]:
q1 = "What does Microsoft cite as a key competitive risk in its 2020 10-K?"
retrieved_q1 = rerank_retrieve("risk_factors_recursive", q1, candidate_k=35, final_k=5)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_q1]}")
answer_q1 = generate_answer(q1, retrieved_q1)
print(f"\nAnswer: {answer_q1}")

Retrieved from: ['Apple', 'Apple', 'Microsoft', 'Apple', 'Microsoft']

Answer: Microsoft cites its increasing focus on cloud-based services as a key competitive risk in its 2020 10-K.


In [ ]:
for r in retrieved_q1:
    print(f"[{r.payload['company']}] score={r.score:.4f}")
    print(r.payload["text"][:300])
    print()

[Apple] score=0.0220
The Company has invested, and in the future may invest, in new business strategies or acquisitions. Such endeavors may involve significant risks and uncertainties, including distraction of management from current operations, greater-than-expected liabilities and expenses, economic, political, legal 

[Apple] score=0.0150
The Company’s services also face substantial competition, including from companies that have significant resources and experience and have established service offerings with large customer bases. The Company competes with business models that provide content to users for free. The Company also compe

[Microsoft] score=0.0149
Our increasing focus on cloud-based services presents execution and competitive risks. A growing part of our business involves cloud-based services available across the spectrum of computing devices. Our strategic vision is to compete and grow by building best-in-class platforms and productivity ser

[Apple] score=0.0138
The Co

In [ ]:
# Combines metadata filtering (comparative questions) + widened reranking
# (single-company questions) — the two techniques that actually moved
# your score today.
def final_retrieve(collection_name, query, candidate_k=35, final_k=5):
    companies = detect_companies(query, COMPANY_NAMES)
    if len(companies) >= 2:
        return retrieve_with_decomposition(collection_name, query, top_k=3)
    else:
        return rerank_retrieve(collection_name, query, candidate_k, final_k)

# Run the full eval set with this combined retrieval strategy
final_results = []
for item in eval_set:
    retrieved = final_retrieve("risk_factors_recursive", item["question"])
    answer = generate_answer(item["question"], retrieved)
    final_results.append({
        "id": item["id"], "type": item["type"], "question": item["question"],
        "expected_answer": item["expected_answer"], "generated_answer": answer,
        "retrieved_companies": [r.payload["company"] for r in retrieved]
    })
    print(f"Q{item['id']} [{item['type']}]: {item['question']}")
    print(f"  Generated: {answer[:200]}")
    print(f"  Retrieved from: {[r.payload['company'] for r in retrieved]}")
    print()

with open("final_eval_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print(f"Saved {len(final_results)} results")

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: Microsoft cites its increasing focus on cloud-based services as a key competitive risk in its 2020 10-K.
  Retrieved from: ['Apple', 'Apple', 'Microsoft', 'Apple', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: Over 80% of total revenues from the display of ads online in 2020.
  Retrieved from: ['Alphabet (Google)', 'Meta (Facebook)', 'Alphabet (Google)', 'Nvidia', 'Nvidia']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: Panasonic
  Retrieved from: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)', 'Tesla', 'Tesla']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple', 'Nvidia', 'Nvidia']

Q5 [fac

In [ ]:
#Compaction
def inspect_for_compaction(collection_name, eval_set):
    """
    For every eval question, retrieve using our final pipeline and check:
    1. Are any retrieved chunks near-duplicates of each other?
    2. Does the company match what the question is actually about?
    3. What order are chunks in before our reorder-by-score step runs?
    """
    for item in eval_set:
        retrieved = final_retrieve(collection_name, item["question"])
        print(f"Q{item['id']}: {item['question']}")

        texts = [r.payload["text"] for r in retrieved]
        companies = [r.payload["company"] for r in retrieved]

        # Crude duplicate check: flag chunks that share a long substring
        # (a cheap proxy for "these are likely overlapping/near-duplicate
        # chunks," common when chunk_overlap causes adjacent chunks to
        # repeat content).
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                overlap_len = len(set(texts[i][:200].split()) & set(texts[j][:200].split()))
                if overlap_len > 15:  # rough threshold, tune by eye
                    print(f"  ⚠️ Possible duplicate: chunk {i+1} and chunk {j+1} share {overlap_len} words")

        print(f"  Companies retrieved: {companies}")
        print(f"  Scores (pre-reorder): {[round(r.score, 4) for r in retrieved]}")
        print()

inspect_for_compaction("risk_factors_recursive", eval_set)

Q1: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Companies retrieved: ['Apple', 'Apple', 'Microsoft', 'Apple', 'Microsoft']
  Scores (pre-reorder): [np.float32(0.022), np.float32(0.015), np.float32(0.0149), np.float32(0.0138), np.float32(0.0093)]

Q2: What percentage of Alphabet's 2020 revenue came from advertising?
  Companies retrieved: ['Alphabet (Google)', 'Meta (Facebook)', 'Alphabet (Google)', 'Nvidia', 'Nvidia']
  Scores (pre-reorder): [np.float32(0.5641), np.float32(0.1853), np.float32(0.1229), np.float32(0.0166), np.float32(0.012)]

Q3: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Companies retrieved: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)', 'Tesla', 'Tesla']
  Scores (pre-reorder): [np.float32(0.0323), np.float32(0.026), np.float32(0.0202), np.float32(0.0156), np.float32(0.0082)]

Q4: What four markets does Nvidia say its GPU-based computing platforms address?
  Companies retrieved: ['Nvid

In [ ]:
def evaluate_retrieval_metrics(eval_set, collection_name):
    """
    Classic IR metrics — no LLM calls, fully deterministic, just math
    over your retrieved results vs known-correct companies/sections.
    """
    results_table = []

    for item in eval_set:
        retrieved = final_retrieve(collection_name, item["question"])
        retrieved_companies = [r.payload["company"] for r in retrieved]

        # The "correct" companies for this question, from your eval set's
        # company field (comma-separated for comparative questions).
        correct_companies = [c.strip() for c in item.get("company", "").split(",") if c.strip() and c.strip() != "N/A"]

        if not correct_companies:
            # Negative/out-of-scope questions have no "correct" company to find
            results_table.append({"id": item["id"], "type": item["type"], "recall_at_k": None, "precision_at_k": None})
            continue

        # Recall@k: of the companies we NEEDED, how many actually appeared
        # somewhere in the retrieved results? (1.0 = found everything needed)
        found = [c for c in correct_companies if c in retrieved_companies]
        recall_at_k = len(found) / len(correct_companies)

        # Precision@k: of what we retrieved, how much was actually from a
        # correct/relevant company? (1.0 = no wasted/irrelevant retrieval)
        relevant_retrieved = [c for c in retrieved_companies if c in correct_companies]
        precision_at_k = len(relevant_retrieved) / len(retrieved_companies)

        # MRR component: rank position (1-indexed) of the FIRST correct
        # company in the retrieved list — rewards getting it near the top.
        first_correct_rank = None
        for i, c in enumerate(retrieved_companies):
            if c in correct_companies:
                first_correct_rank = i + 1
                break
        reciprocal_rank = 1 / first_correct_rank if first_correct_rank else 0

        results_table.append({
            "id": item["id"], "type": item["type"],
            "recall_at_k": round(recall_at_k, 3),
            "precision_at_k": round(precision_at_k, 3),
            "reciprocal_rank": round(reciprocal_rank, 3)
        })

    return results_table


metrics_results = evaluate_retrieval_metrics(eval_set, "risk_factors_recursive")
metrics_df = pd.DataFrame(metrics_results)
print(metrics_df)

# Aggregate scores across questions that HAD a correct company to find
scoreable = metrics_df.dropna(subset=["recall_at_k"])
print(f"\nMean Recall@k: {scoreable['recall_at_k'].mean():.3f}")
print(f"Mean Precision@k: {scoreable['precision_at_k'].mean():.3f}")
print(f"Mean Reciprocal Rank (MRR): {scoreable['reciprocal_rank'].mean():.3f}")

metrics_df.to_csv("retrieval_metrics.csv", index=False)

   id         type  recall_at_k  precision_at_k  reciprocal_rank
0   1      factual          1.0             0.4            0.333
1   2      factual          1.0             0.4            1.000
2   3      factual          1.0             0.4            0.250
3   4      factual          1.0             0.8            1.000
4   5      factual          1.0             1.0            1.000
5   6  comparative          1.0             1.0            1.000
6   7  comparative          0.5             0.5            1.000
7   8     negative          0.0             0.0            0.000
8   9     negative          1.0             0.4            0.500

Mean Recall@k: 0.833
Mean Precision@k: 0.544
Mean Reciprocal Rank (MRR): 0.676


In [ ]:
#!pip uninstall ragas langchain langchain-community langchain-openai -y
!pip install ragas==0.1.9 langchain==0.2.16 langchain-community==0.2.16 langchain-openai==0.1.23

Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: langchain 1.3.10
Uninstalling langchain-1.3.10:
  Successfully uninstalled langchain-1.3.10
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-openai 1.3.2
Uninstalling langchain-openai-1.3.2:
  Successfully uninstalled langchain-openai-1.3.2
     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     --- ------------------------------------ 1.6/15.8 MB 10.9 MB/s eta 0:00:02
     ------------ --------------------------- 5.0/15.8 MB 14.7 MB/s eta 0:00:01
     --------------------- ------------------ 8.4/15.8 MB 15.2 MB/s eta 0:00:01
     ----------------------------- --------- 11.8/15.8 MB 15.7 MB/s eta 0:00:01
     --------------------------------------  15.7/15.8 MB 16.7 MB/s eta 0:00:01
     ---------

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\preet\anaconda3\python.exe C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f\vendored-meson\meson\meson.py setup C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f\.mesonpy-k77g7wk8 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f\.mesonpy-k77g7wk8\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f
      Build dir: C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97

In [60]:
#alternate check faithfulness directly 
def check_faithfulness(question, answer, contexts):
    """
    A simple custom faithfulness check — no RAGAS dependency needed.
    Asks the LLM directly: is this answer supported by this context?
    """
    context_text = "\n\n".join(contexts)
    prompt = f"""Context:
{context_text}

Answer: {answer}

Is the above answer fully supported by the context, with no unsupported claims? Reply with only "YES" or "NO" followed by a one-sentence reason."""

    response = llm.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    return response.choices[0].message.content

for item in eval_set:
    retrieved = final_retrieve("risk_factors_recursive", item["question"])
    answer = generate_answer(item["question"], retrieved)
    contexts = [r.payload["text"] for r in retrieved]
    verdict = check_faithfulness(item["question"], answer, contexts)
    print(f"Q{item['id']}: {verdict}")

Q1: NO - The context provided is from Apple Inc.'s 2020 Form 10-K, not Microsoft, and discusses Apple's focus on cloud-based services as a competitive risk.
Q2: YES, the answer is fully supported by the context, as it explicitly states that "We generated over 80% of total revenues from the display of ads online in 2020."
Q3: YES, the answer is supported by the context as it mentions Panasonic specifically in relation to the manufacturing of battery cells for the company's products at the Gigafactory Nevada.
Q4: YES, the answer is fully supported as it directly lists the four large markets addressed by the company's GPU-based visual and accelerated computing platforms mentioned in the context.
Q5: YES, the answer is fully supported by the context as it directly references the recommendation to read specific sections of the Form 10-K for a comprehensive understanding of the risk factors discussed.
Q6: YES. The answer accurately reflects the distinct risks related to COVID-19 as described

What classic IR metrics (recall@k, precision@k, MRR) can and can't tell you: they only measure retrieval — did the right source material get pulled in. They have zero ability to judge the final generated answer. You could have perfect recall@k (the right chunk was retrieved) and still get a wrong or hallucinated answer if the LLM ignores or misreads that chunk — exactly what happened with your Alphabet "80%" case, where retrieval succeeded but generation initially failed. A pure IR metric would have scored that case as a success, even though the actual answer was wrong. That's a real blind spot.
What LLM-judge metrics (faithfulness, answer relevance) add: they look at the actual generated text and ask "does this answer's claims trace back to the retrieved context" and "does this answer actually address the question" — judgments that require genuine language understanding, not just string/ID matching. There's no deterministic formula for "is this sentence faithful to that paragraph" — you need something that can read and reason, which is why an LLM judge is used despite the downsides.

**Summary**

1. Naive baseline (fixed-size chunks, dense-only retrieval) → establish "before" score

2. Better chunking (recursive/semantic) → fixes malformed chunks 
   (foundation — every later step depends on chunk quality)

3. Metadata filtering / query decomposition → fixes structural retrieval gaps 
   (biggest ROI, fixes WHAT gets retrieved — do before fine-tuning HOW)

4. Hybrid search (BM25 + RRF) → fixes exact-term/lexical misses 
   (only add if diagnosed need — exact terms, codes, names)

5. Query reformulation / HyDE → fixes question-vs-answer vocabulary mismatch 
   (only add if diagnosed need — phrasing gap, not content gap)

6. Reranking → fixes precision among retrieved candidates 
   (needs steps 2-5 done first — reranking can't fix what retrieval never found)

7. Compaction (reorder, dedupe) → fixes how the LLM uses what's retrieved 
   (last — polishes already-correct retrieval, doesn't fix bad retrieval)